# UTSM: retrain with Laval data
Upload this notebook to Colab, select **CPU**, then **Run all**. Upload **Laval All Telem.zip** when prompted.

**No restart loop:** training, data preparation and model inference run in separate Python processes.
This notebook does not import NumPy, pandas or scikit-learn into the interactive kernel.
It therefore works even when Colab has already loaded different versions of those libraries.

Default results remain exploratory: current calibration is unverified and voltage is constant.
The workflow reports whether the forest improves on the analytical baseline; improvement is not assumed.


## 1. Prepare the worker environment
The pinned Python 3.10-3.13 packages are checked in a fresh Python process.
Existing NumPy/pandas objects in Colab's kernel are left alone. **No session restart is required.**


In [ ]:
DEPENDENCIES_READY = False
import os, sys, subprocess, importlib.util, json, hashlib
from pathlib import Path
REQUIREMENTS = 'numpy==2.2.6\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\njoblib==1.4.2\n'
BUNDLED_ENVIRONMENT_SOURCE = '"""Standard-library-only environment setup for Colab\'s in-process Python kernel."""\nfrom __future__ import annotations\n\nimport importlib\nimport importlib.metadata\nimport json\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nMODULES = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",\n           "scikit-learn": "sklearn", "matplotlib": "matplotlib", "joblib": "joblib"}\nRESTART_MESSAGE = (\n    "Packages on disk are ready, but this kernel still contains old scientific-library binaries. "\n    "In Colab choose Runtime > Restart session (or Restart runtime), then Run all again. "\n    "Do not rerun only the model-loading cell."\n)\n\n\ndef parse_pins(text):\n    pins = {}\n    for line in text.splitlines():\n        line = line.strip()\n        if not line or line.startswith("#"):\n            continue\n        name, sep, version = line.partition("==")\n        if not sep or name not in MODULES or not version:\n            raise ValueError("Expected supported exact dependency pins; got: " + line)\n        pins[name] = version\n    return pins\n\n\ndef installed_versions(pins):\n    versions = {}\n    for package in pins:\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = None\n    return versions\n\n\ndef loaded_versions(pins):\n    return {package: vars(sys.modules[MODULES[package]]).get("__version__")\n            for package in pins if MODULES[package] in sys.modules}\n\n\ndef requires_restart(pins, before, after, loaded, repaired=False):\n    # Includes partially imported packages, which can lack __version__.\n    return any(repaired or before.get(package) != after.get(package)\n               or loaded_version != pins[package]\n               for package, loaded_version in loaded.items())\n\n\ndef validate_kernel(pins):\n    try:\n        for package, expected in pins.items():\n            module = importlib.import_module(MODULES[package])\n            actual = getattr(module, "__version__", None)\n            if actual != expected:\n                raise RuntimeError(f"{package}: loaded {actual}, expected {expected}")\n        import numpy as np\n        import pandas as pd\n        from sklearn.linear_model import Ridge\n        # Exercise the random C extension from the user\'s failing traceback.\n        np.random.default_rng(0).normal(size=2)\n        np.random.RandomState(0).normal(size=2)\n        data = pd.DataFrame({"x": [0.0, 1.0, 2.0]})\n        Ridge().fit(data, [0.0, 1.0, 2.0]).predict(data)\n    except Exception as exc:\n        raise RuntimeError(RESTART_MESSAGE + "\\nKernel check: " + str(exc)) from exc\n\n\ndef probe_fresh_process(pins):\n    probe = """\nimport importlib, json, sys\npins = json.loads(sys.argv[1])\nmodules = json.loads(sys.argv[2])\nfor package, expected in pins.items():\n    module = importlib.import_module(modules[package])\n    assert module.__version__ == expected, (package, module.__version__, expected)\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import Ridge\nnp.random.default_rng(0).normal(size=2)\nnp.random.RandomState(0).normal(size=2)\nx = pd.DataFrame({\'x\': [0., 1., 2.]})\nRidge().fit(x, [0., 1., 2.]).predict(x)\nprint(\'Fresh-process scientific import and binary checks passed.\')\n"""\n    return subprocess.run([sys.executable, "-c", probe, json.dumps(pins), json.dumps(MODULES)],\n                          text=True, capture_output=True)\n\n\ndef ensure_environment(requirements_path, validate_active_kernel=True):\n    if not (3, 10) <= sys.version_info[:2] <= (3, 13):\n        raise RuntimeError("This pinned notebook supports Python 3.10-3.13; choose a supported Colab runtime.")\n    path = Path(requirements_path)\n    pins = parse_pins(path.read_text(encoding="utf-8"))\n    before = installed_versions(pins)\n    # Check sys.modules only; never import a binary package before installation.\n    loaded = loaded_versions(pins) if validate_active_kernel else {}\n    command = [sys.executable, "-m", "pip", "install", "--upgrade", "--only-binary=:all:",\n               "--no-cache-dir", "-r", str(path)]\n    changed = before != pins\n    if changed:\n        subprocess.check_call(command)\n    importlib.invalidate_caches()\n    probe = probe_fresh_process(pins)\n    repaired = False\n    if probe.returncode:\n        # Metadata can claim correct versions while partial installs contain stale binaries.\n        subprocess.check_call(command + ["--force-reinstall"])\n        repaired = True\n        importlib.invalidate_caches()\n        probe = probe_fresh_process(pins)\n    if probe.returncode:\n        raise RuntimeError("Scientific package checks failed even in a fresh Python process. "\n                           "Use a clean Colab runtime and rerun installation.\\n" + probe.stderr[-4000:])\n    after = installed_versions(pins)\n    if after != pins:\n        raise RuntimeError("Installed versions do not match the notebook pins: " + str(after))\n    if validate_active_kernel and requires_restart(pins, before, after, loaded, repaired):\n        raise RuntimeError(RESTART_MESSAGE)\n    # Catch partially loaded NumPy internals even if __version__ appears correct.\n    if validate_active_kernel:\n        validate_kernel(pins)\n    else:\n        print("Worker environment ready. Notebook scientific libraries will not be imported or restarted.")\n    print(probe.stdout.strip())\n    print("Python:", sys.version.split()[0])\n    print("Ready:", json.dumps(after, sort_keys=True))\n    return after\n'
IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
WORK_DIR = Path(os.environ.get("UTSM_WORK_DIR", "/content/utsm-laval" if IN_COLAB else "laval-runtime")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
requirements_path = WORK_DIR/"requirements-colab.txt"
requirements_path.write_text(REQUIREMENTS, encoding="utf-8")
environment_path = WORK_DIR/"colab_environment.py"
environment_path.write_text(BUNDLED_ENVIRONMENT_SOURCE, encoding="utf-8")
env_spec = importlib.util.spec_from_file_location("utsm_colab_environment", environment_path)
environment = importlib.util.module_from_spec(env_spec)
sys.modules[env_spec.name] = environment
env_spec.loader.exec_module(environment)
environment.ensure_environment(requirements_path, validate_active_kernel=False)
DEPENDENCIES_READY = True
print("Workspace:", WORK_DIR)


## 2. Write the bundled implementation
The notebook embeds the training source, worker and analytical reference. It needs no GitHub login or source download.
The model code is written to disk and loaded only by the separate worker.


In [ ]:
if not globals().get('DEPENDENCIES_READY', False):
    raise RuntimeError('Run the dependency cell successfully first.')
BUNDLED_TRAINING_SOURCE = '"""Reproducible, session-held-out Laval power-model training.\n\nPython 3.9+; see requirements-colab.txt for the Colab environment.\nRaw archives are read in memory, never extracted or modified.\nThis is an offline power/energy estimator, not a throttle controller.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nimport subprocess\nimport sys\nimport zipfile\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nSCHEMA_VERSION = "laval-power-v1"\nCORE_FEATURES = [\n    "speed_kph", "speed_squared", "positive_accel", "negative_accel",\n    "positive_accel_speed", "negative_accel_speed",\n]\nENHANCED_FEATURES = CORE_FEATURES + [\n    "temperature_C", "temperature_missing", "speed_mean_3", "speed_std_3",\n    "wheel_speed_used",\n]\nREQUIRED = {"timestamp_ms", "current_mA", "voltage_mV"}\nCANDIDATES = [\n    {"name": "rf_core_leaf20", "features": "core", "min_samples_leaf": 20, "max_depth": 12},\n    {"name": "rf_enhanced_leaf20", "features": "enhanced", "min_samples_leaf": 20, "max_depth": 12},\n    {"name": "rf_enhanced_leaf5", "features": "enhanced", "min_samples_leaf": 5, "max_depth": 12},\n    {"name": "rf_enhanced_leaf50", "features": "enhanced", "min_samples_leaf": 50, "max_depth": None},\n]\n\n\n@dataclass\nclass Config:\n    # These defaults are explicitly exploratory, not a sensor calibration claim.\n    current_mode: str = "magnitude"  # magnitude | positive | negative\n    current_scale: float = 1.0\n    current_zero_mA: float = 0.0\n    calibration_confirmed: bool = False\n    calibration_overrides: dict[str, dict[str, Any]] = field(default_factory=dict)\n    test_dates: list[str] = field(default_factory=lambda: ["2026-09-20"])\n    max_gap_s: float = 2.0\n    min_speed_kph: float = 2.0\n    max_speed_kph: float = 60.0\n    max_abs_accel_m_s2: float = 4.0\n    max_abs_current_mA: float = 60000.0\n    max_hdop: float = 3.0\n    max_speed_disagreement_kph: float = 10.0\n    min_session_rows: int = 50\n    min_session_duration_s: float = 20.0\n    window_seconds: float = 10.0\n    min_window_duration_s: float = 5.0\n    n_estimators: int = 200\n    n_jobs: int = 2\n    seed: int = 42\n    refit_all_for_release: bool = False\n\n    def validate(self):\n        if self.current_mode not in {"magnitude", "positive", "negative"}:\n            raise ValueError("current_mode must be magnitude, positive or negative")\n        if not np.isfinite(self.current_scale) or self.current_scale <= 0:\n            raise ValueError("current_scale must be positive and finite")\n        if not np.isfinite(self.current_zero_mA):\n            raise ValueError("current_zero_mA must be finite")\n        if not self.test_dates or any(not re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", d) for d in self.test_dates):\n            raise ValueError("Provide explicit ISO test_dates before training")\n        positive = [self.max_gap_s, self.max_speed_kph, self.max_abs_accel_m_s2,\n                    self.max_abs_current_mA, self.max_hdop, self.max_speed_disagreement_kph,\n                    self.min_session_duration_s, self.window_seconds, self.min_window_duration_s]\n        if not all(np.isfinite(v) and v > 0 for v in positive):\n            raise ValueError("Time, speed and quality limits must be positive and finite")\n        if not 0 <= self.min_speed_kph < self.max_speed_kph:\n            raise ValueError("Invalid speed range")\n        if self.min_session_rows < 2 or self.n_estimators < 1 or self.n_jobs < 1:\n            raise ValueError("Invalid sample or forest configuration")\n        if self.min_window_duration_s > self.window_seconds:\n            raise ValueError("min_window_duration_s cannot exceed window_seconds")\n        if self.refit_all_for_release and not self.calibration_confirmed:\n            raise ValueError("Confirm calibration before requesting a release refit")\n\n\ndef sha256(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef number(df: pd.DataFrame, name: str, default=np.nan) -> pd.Series:\n    if name not in df:\n        return pd.Series(default, index=df.index, dtype=float)\n    return pd.to_numeric(df[name], errors="coerce")\n\n\ndef calibrated_current(values: pd.Series, cfg: Config, filename: str) -> pd.Series:\n    override = cfg.calibration_overrides.get(filename, {})\n    unknown = set(override) - {"current_mode", "current_scale", "current_zero_mA"}\n    if unknown:\n        raise ValueError(f"Unknown calibration keys for {filename}: {sorted(unknown)}")\n    mode = override.get("current_mode", cfg.current_mode)\n    scale = float(override.get("current_scale", cfg.current_scale))\n    zero = float(override.get("current_zero_mA", cfg.current_zero_mA))\n    if mode not in {"magnitude", "positive", "negative"} or not np.isfinite(scale) or scale <= 0 or not np.isfinite(zero):\n        raise ValueError(f"Invalid calibration for {filename}")\n    corrected = (values - zero) * scale\n    if mode == "magnitude":\n        return corrected.abs()\n    # Consumption only: regen is not silently counted as positive consumption.\n    return (corrected if mode == "positive" else -corrected).clip(lower=0)\n\n\ndef causal_features(df: pd.DataFrame) -> pd.DataFrame:\n    """Rows must be in source order, with blocks reset after outages/reboots."""\n    out = df.copy()\n    groups = out.groupby(["session_id", "block_id"], sort=False)\n    smooth = groups["raw_speed_kph"].transform(\n        lambda s: s.rolling(3, min_periods=1).median()\n    )\n    out["speed_kph"] = smooth\n    prev_speed = smooth.groupby([out.session_id, out.block_id], sort=False).shift()\n    out["accel_m_s2"] = (smooth - prev_speed) / (3.6 * out["dt_s"])\n    speed_groups = smooth.groupby([out.session_id, out.block_id], sort=False)\n    out["speed_mean_3"] = speed_groups.transform(lambda s: s.rolling(3, min_periods=1).mean())\n    out["speed_std_3"] = speed_groups.transform(lambda s: s.rolling(3, min_periods=1).std(ddof=0))\n    out["speed_squared"] = smooth ** 2\n    out["positive_accel"] = out.accel_m_s2.clip(lower=0)\n    out["negative_accel"] = (-out.accel_m_s2).clip(lower=0)\n    out["positive_accel_speed"] = out.positive_accel * smooth\n    out["negative_accel_speed"] = out.negative_accel * smooth\n    out["temperature_missing"] = out.temperature_C.isna().astype(float)\n    return out\n\n\ndef prepare_session(raw: pd.DataFrame, filename: str, cfg: Config) -> tuple[pd.DataFrame, dict]:\n    """Build valid observed-driving intervals without bridging invalid rows."""\n    missing = REQUIRED - set(raw.columns)\n    if missing:\n        raise ValueError(f"{filename}: missing columns {sorted(missing)}")\n    df = raw.copy().reset_index(drop=True)\n    stamp = re.match(r"(\\d{4}-\\d{2}-\\d{2})", Path(filename).name)\n    if not stamp:\n        raise ValueError(f"{filename}: date must begin YYYY-MM-DD")\n    session = Path(filename).stem\n    df["session_id"] = session\n    df["date"] = stamp.group(1)\n    df["source_row"] = np.arange(len(df))\n    ts = number(df, "timestamp_ms")\n    current_raw = number(df, "current_mA")\n    current = calibrated_current(current_raw, cfg, Path(filename).name)\n    voltage = number(df, "voltage_mV") / 1000.0\n    dt = ts.diff() / 1000.0\n    temporal = dt.gt(0) & dt.le(cfg.max_gap_s) & ts.notna() & ts.shift().notna()\n\n    wheel = number(df, "wheel_speed_kmph")\n    if "wheel_speed_kph" in df:\n        wheel = number(df, "wheel_speed_kph")\n    gps = number(df, "gps_speed_kmph")\n    wheel_ok = number(df, "wheel_speed_valid", 0).eq(1) & wheel.between(0, cfg.max_speed_kph)\n    lat = number(df, "gps_lat")\n    lon = number(df, "gps_long")\n    gps_ok = (number(df, "gps_location_valid", 0).eq(1)\n              & gps.between(0, cfg.max_speed_kph)\n              & lat.between(-90, 90) & lon.between(-180, 180)\n              & (lat.abs() + lon.abs()).gt(0)\n              & number(df, "gps_hdop").between(0, cfg.max_hdop))\n    if "gps_time_valid" in df:\n        gps_ok &= number(df, "gps_time_valid", 0).eq(1)\n    if "gps_time_utc" in df:\n        clock = df["gps_time_utc"].astype(str)\n        if "gps_centisecond" in df:\n            clock = clock + ":" + df["gps_centisecond"].astype(str)\n        clock_changed = clock.ne(clock.shift()) | dt.le(0)\n        last_clock_update_ms = ts.where(clock_changed).ffill()\n        gps_ok &= (ts - last_clock_update_ms).between(0, cfg.max_gap_s * 1000)\n    # Reject disagreement instead of choosing the sensor that makes a model win.\n    disagreement = wheel_ok & gps_ok & (wheel - gps).abs().gt(cfg.max_speed_disagreement_kph)\n    speed = wheel.where(wheel_ok, gps.where(gps_ok)).mask(disagreement)\n    temperature = number(df, "temperature_C")\n    if "motor_temperature_C" in df:\n        temperature = number(df, "motor_temperature_C")\n    temp_flag = "motor_temperature_valid" if "motor_temperature_valid" in df else "temperature_valid"\n    temperature = temperature.where(number(df, temp_flag, 0).eq(1) & temperature.between(-40, 180))\n    electrical = (current_raw.abs().le(cfg.max_abs_current_mA)\n                  & voltage.between(18, 30) & current.notna())\n    sensor_ok = electrical & speed.notna()\n    # Every invalid reading or timestamp discontinuity begins a new causal block.\n    boundaries = (~temporal) | (~sensor_ok) | (~sensor_ok.shift(fill_value=False))\n    block = boundaries.cumsum()\n    df["block_id"] = block\n    df["time_s"] = ts / 1000.0\n    df["dt_s"] = dt\n    df["raw_speed_kph"] = speed\n    df["temperature_C"] = temperature\n    df["wheel_speed_used"] = wheel_ok.astype(float)\n    df["corrected_current_mA"] = current\n    df["voltage_v"] = voltage\n    power = current / 1000.0 * voltage\n    # Target is interval-average power; integration uses the same intervals for all models.\n    df["target_power_w"] = (power + power.shift()) / 2.0\n    valid_interval = temporal & sensor_ok & sensor_ok.shift(fill_value=False)\n    df = causal_features(df)\n    valid = (valid_interval & df.speed_kph.between(cfg.min_speed_kph, cfg.max_speed_kph)\n             & df.accel_m_s2.abs().le(cfg.max_abs_accel_m_s2))\n    df["actual_energy_j"] = df.target_power_w * df.dt_s\n    finite = np.isfinite(df[CORE_FEATURES + ["target_power_w", "dt_s"]].to_numpy()).all(axis=1)\n    valid &= finite\n    kept = df.loc[valid].copy()\n    # Separate report groups at every excluded interval; no invented whole laps.\n    continuity = kept.source_row.diff().ne(1) | kept.block_id.diff().ne(0)\n    kept["evaluation_block"] = continuity.cumsum().astype(int)\n    report = {\n        "filename": filename, "session_id": session, "date": stamp.group(1),\n        "raw_rows": len(raw), "accepted_rows": len(kept),\n        "accepted_duration_s": float(kept.dt_s.sum()),\n        "file_span_s": float((ts.max() - ts.min()) / 1000) if ts.notna().any() else 0,\n        "negative_current_rows": int(current_raw.lt(0).sum()),\n        "gps_valid_rows": int(gps_ok.sum()), "wheel_valid_rows": int(wheel_ok.sum()),\n        "speed_disagreement_rows": int(disagreement.sum()),\n        "timestamp_reset_or_duplicate_rows": int(dt.le(0).sum()),\n        "gap_rows": int(dt.gt(cfg.max_gap_s).sum()),\n        "constant_voltage": bool(voltage.nunique(dropna=True) <= 1),\n        "voltage_median_v": float(voltage.median()) if voltage.notna().any() else None,\n        "status": "accepted",\n    }\n    if len(kept) < cfg.min_session_rows or kept.dt_s.sum() < cfg.min_session_duration_s:\n        report["status"] = "excluded_insufficient_valid_driving"\n        report["eligible_rows_before_session_filter"] = len(kept)\n        report["accepted_rows"] = 0\n        report["accepted_duration_s"] = 0.0\n        kept = kept.iloc[:0]\n    return kept, report\n\n\ndef load_archive(archive: str | Path, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, dict]:\n    cfg.validate()\n    archive = Path(archive)\n    if archive.stat().st_size > 500_000_000:\n        raise ValueError("Archive exceeds 500 MB; split the archive before training")\n    source = archive.read_bytes()\n    if len(source) > 500_000_000:\n        raise ValueError("Archive exceeds 500 MB; split the archive before training")\n    frames, reports, hashes = [], [], {}\n    seen, seen_rows, session_ids = set(), set(), set()\n    with zipfile.ZipFile(io.BytesIO(source)) as z:\n        infos = [i for i in z.infolist() if i.filename.lower().endswith(".csv") and not i.is_dir()\n                 and not any(p.startswith(".") or p == "__MACOSX" for p in Path(i.filename).parts)]\n        if sum(i.file_size for i in infos) > 1_000_000_000:\n            raise ValueError("Uncompressed CSV contents exceed 1 GB")\n        for info in sorted(infos, key=lambda i: i.filename):\n            data = z.read(info)  # no path extraction\n            normalized = data.decode("utf-8-sig").replace("\\r\\n", "\\n").replace("\\r", "\\n")\n            digest = sha256(normalized.encode())\n            hashes[info.filename] = {"sha256": sha256(data), "normalized_sha256": digest, "bytes": len(data)}\n            if digest in seen:\n                reports.append({"filename": info.filename, "status": "excluded_duplicate_content"})\n                continue\n            seen.add(digest)\n            try:\n                raw = pd.read_csv(io.StringIO(normalized))\n            except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:\n                reports.append({"filename": info.filename, "status": "excluded_unreadable", "reason": str(exc)})\n                continue\n            semantic = sha256(json.dumps(raw.columns.tolist()).encode() + pd.util.hash_pandas_object(raw, index=False).to_numpy().tobytes())\n            hashes[info.filename]["parsed_rows_sha256"] = semantic\n            if semantic in seen_rows:\n                reports.append({"filename": info.filename, "status": "excluded_duplicate_content"})\n                continue\n            seen_rows.add(semantic)\n            if not REQUIRED.issubset(raw.columns):\n                reports.append({"filename": info.filename, "status": "excluded_missing_columns",\n                                "reason": str(sorted(REQUIRED - set(raw.columns)))})\n                continue\n            frame, report = prepare_session(raw, info.filename, cfg)\n            if report["session_id"] in session_ids:\n                raise ValueError("Duplicate session basename in archive: " + report["session_id"])\n            session_ids.add(report["session_id"])\n            reports.append(report)\n            if not frame.empty:\n                frames.append(frame)\n    unknown_overrides = set(cfg.calibration_overrides) - {Path(n).name for n in hashes}\n    if unknown_overrides:\n        raise ValueError("Calibration overrides name missing CSVs: " + str(sorted(unknown_overrides)))\n    if not frames:\n        raise ValueError("No usable driving intervals. Inspect calibration, sensor flags and data schema.")\n    frame = pd.concat(frames, ignore_index=True)\n    if frame[["session_id", "source_row"]].duplicated().any():\n        raise AssertionError("Duplicate source rows")\n    # A reboot must not create a separate training/test group for the same CSV.\n    provenance = {"archive": archive.name, "archive_sha256": sha256(source), "files": hashes}\n    return frame, pd.DataFrame(reports), provenance\n\n\ndef split_sessions(frame: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, dict]:\n    test = frame[frame.date.isin(cfg.test_dates)].copy()\n    dev = frame[~frame.date.isin(cfg.test_dates)].copy()\n    if test.empty:\n        raise ValueError("No valid test sessions on configured test_dates")\n    if dev.session_id.nunique() < 2:\n        raise ValueError("Need at least two usable development sessions for grouped tuning")\n    if dev.date.max() >= test.date.min():\n        raise ValueError("Development dates must precede test dates; do not randomly split rows")\n    if set(dev.session_id) & set(test.session_id):\n        raise AssertionError("Session leakage")\n    # Exact numeric rows can recur naturally at rest; duplicate timestamps/exports need stronger checks.\n    # Identical sequences across files are excluded above by normalized content hash.\n    manifest = {"development_sessions": sorted(dev.session_id.unique().tolist()),\n                "test_sessions": sorted(test.session_id.unique().tolist()),\n                "test_dates": cfg.test_dates, "development_rows": len(dev), "test_rows": len(test)}\n    return dev, test, manifest\n\n\ndef sample_weights(frame: pd.DataFrame) -> np.ndarray:\n    # Equal total weight per session, distributed by actual observed duration.\n    session_seconds = frame.groupby("session_id").dt_s.transform("sum").to_numpy()\n    weights = frame.dt_s.to_numpy() / session_seconds\n    return weights * len(frame) / frame.session_id.nunique()\n\n\ndef feature_names(kind: str) -> list[str]:\n    if kind == "core":\n        return list(CORE_FEATURES)\n    if kind == "enhanced":\n        return list(ENHANCED_FEATURES)\n    raise ValueError(f"Unknown feature set: {kind}")\n\n\ndef train_forest(frame: pd.DataFrame, candidate: dict, cfg: Config) -> dict:\n    names = feature_names(candidate["features"])\n    estimator = make_pipeline(\n        SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),\n        RandomForestRegressor(n_estimators=cfg.n_estimators, random_state=cfg.seed,\n                              n_jobs=cfg.n_jobs, min_samples_leaf=candidate["min_samples_leaf"],\n                              max_depth=candidate["max_depth"]),\n    )\n    estimator.fit(frame[names], frame.target_power_w,\n                  randomforestregressor__sample_weight=sample_weights(frame))\n    return {"schema_version": SCHEMA_VERSION, "kind": "random_forest", "features": names,\n            "estimator": estimator, "candidate": candidate}\n\n\ndef train_analytical(frame: pd.DataFrame) -> dict:\n    # A matched-data analytical refit; distinct from the frozen Front Campus deployment artifact.\n    # Standardization parameters are learned only from this fold.\n    estimator = make_pipeline(StandardScaler(), Ridge(alpha=1.0))\n    estimator.fit(frame[CORE_FEATURES], frame.target_power_w,\n                  ridge__sample_weight=sample_weights(frame))\n    return {"schema_version": SCHEMA_VERSION, "kind": "analytical_refit",\n            "features": list(CORE_FEATURES), "estimator": estimator}\n\n\ndef predict_power(bundle: dict, feature_frame: pd.DataFrame) -> np.ndarray:\n    """Predict interval power from canonical causal features; no label columns required."""\n    if bundle.get("schema_version") != SCHEMA_VERSION:\n        raise ValueError("Unsupported feature schema")\n    missing = set(bundle["features"]) - set(feature_frame.columns)\n    if missing:\n        raise ValueError("Missing prediction features: " + str(sorted(missing)))\n    return np.maximum(bundle["estimator"].predict(feature_frame[bundle["features"]]), 0.0)\n\n\ndef predict_front_campus(frame: pd.DataFrame, reference: dict) -> np.ndarray:\n    """Frozen historical model electrical rule, no motor physics floor, neutral position.\n\n    Historical acceleration-action threshold is 1.0 in classify_strategy_action;\n    preserve it for the frozen reference rather than silently changing the incumbent.\n    Grade is zero: unreliable GPS altitude is not treated as surveyed grade.\n    """\n    speed = frame.speed_kph.to_numpy()\n    accel = frame.accel_m_s2.to_numpy()\n    pos, neg = np.maximum(accel, 0), np.maximum(-accel, 0)\n    action_acc = accel > 1.0\n    action_coast = accel < -1.0\n    action_hold = ~(action_acc | action_coast)\n    zeros = np.zeros(len(frame))\n    design = np.column_stack([np.ones(len(frame)), speed, speed**2, pos, neg, zeros, zeros,\n                              zeros, action_acc, action_hold, action_coast, pos*speed, neg*speed])\n    raw_current = np.maximum(design @ np.asarray(reference["current_coeffs"]), 0)\n    raw_power = np.maximum(design @ np.asarray(reference["power_coeffs"]), 0)\n    voltage = float(reference["median_voltage_v"])\n    on = np.full(len(frame), float(reference["on_current_mA"]))\n    cruise = float(reference["cruise_current_mA"])\n    duty_acc = np.clip(0.22 + pos*1.8, 0.18, 1.0)\n    on = np.where(action_acc, np.maximum(on, raw_current / np.maximum(duty_acc, 0.05)), on)\n    avg = np.where(action_acc, np.maximum(raw_current, on*duty_acc), np.maximum(raw_current, cruise))\n    on = np.where(action_hold, np.maximum(on, avg*1.35), on)\n    duty_hold = np.clip(avg/np.maximum(on, 1), 0.04, 0.85)\n    avg = np.where(action_hold, on*duty_hold, avg)\n    power = np.maximum(raw_power, avg/1000*voltage)\n    return np.where(action_coast, 0.0, power)\n\n\ndef evaluation(frame: pd.DataFrame, predicted_power: np.ndarray, name: str, cfg: Config):\n    if len(frame) != len(predicted_power):\n        raise ValueError("Prediction count does not match evaluation rows")\n    rows = frame[["session_id", "date", "source_row", "time_s", "block_id",\n                  "evaluation_block", "dt_s", "actual_energy_j", "target_power_w"]].copy()\n    rows["model"] = name\n    rows["predicted_power_w"] = np.asarray(predicted_power)\n    rows["predicted_energy_j"] = rows.predicted_power_w * rows.dt_s\n    rows["absolute_power_error_w"] = (rows.predicted_power_w - rows.target_power_w).abs()\n    grouped = rows.groupby(["model", "session_id", "date"], sort=True)\n    sessions = grouped.agg(actual_energy_j=("actual_energy_j", "sum"),\n                           predicted_energy_j=("predicted_energy_j", "sum"),\n                           valid_duration_s=("dt_s", "sum"),\n                           power_mae_w=("absolute_power_error_w", "mean")).reset_index()\n    denom = sessions.actual_energy_j.replace(0, np.nan)\n    sessions["signed_error_pct"] = (sessions.predicted_energy_j - sessions.actual_energy_j) / denom * 100\n    sessions["absolute_error_pct"] = sessions.signed_error_pct.abs()\n    rows["window"] = np.floor(rows.time_s / cfg.window_seconds).astype(int)\n    windows = rows.groupby(["model", "session_id", "evaluation_block", "window"], sort=True).agg(\n        actual_energy_j=("actual_energy_j", "sum"), predicted_energy_j=("predicted_energy_j", "sum"),\n        valid_duration_s=("dt_s", "sum")).reset_index()\n    windows = windows[windows.valid_duration_s >= cfg.min_window_duration_s].copy()\n    windows["absolute_error_pct"] = ((windows.predicted_energy_j - windows.actual_energy_j).abs()\n                                    / windows.actual_energy_j.replace(0, np.nan) * 100)\n    return sessions, windows, rows\n\n\ndef tune_forest(dev: pd.DataFrame, cfg: Config, candidates: list[dict] | None = None):\n    candidates = candidates or CANDIDATES\n    cv = GroupKFold(n_splits=min(3, dev.session_id.nunique()))\n    folds = list(cv.split(dev, groups=dev.session_id))\n    results = []\n    for candidate in candidates:\n        all_sessions = []\n        for fold, (train_index, val_index) in enumerate(folds):\n            train, val = dev.iloc[train_index], dev.iloc[val_index]\n            assert not set(train.session_id) & set(val.session_id)\n            bundle = train_forest(train, candidate, cfg)\n            metrics, _, _ = evaluation(val, predict_power(bundle, val), candidate["name"], cfg)\n            metrics["fold"] = fold\n            all_sessions.append(metrics)\n        metrics = pd.concat(all_sessions, ignore_index=True)\n        score = float(metrics.absolute_error_pct.mean())\n        if not np.isfinite(score):\n            raise ValueError("No finite development CV score; inspect zero-energy sessions")\n        results.append({"candidate": candidate["name"], "cv_mean_session_error_pct": score,\n                        "features": candidate["features"], "min_samples_leaf": candidate["min_samples_leaf"],\n                        "max_depth": candidate["max_depth"]})\n    result = pd.DataFrame(results).sort_values(["cv_mean_session_error_pct", "candidate"]).reset_index(drop=True)\n    selected = next(c for c in candidates if c["name"] == result.iloc[0].candidate)\n    return selected, result\n\n\ndef save_report(out: Path, summary: dict, sessions: pd.DataFrame):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    pivot = sessions.pivot(index="session_id", columns="model", values="absolute_error_pct")\n    ax = pivot.plot.bar(figsize=(11, 5), ylabel="Absolute covered-driving energy error (%)",\n                        title="Held-out Laval sessions (lower is better)")\n    ax.set_xlabel("Session")\n    plt.xticks(rotation=25, ha="right")\n    plt.tight_layout()\n    plt.savefig(out / "comparison.png", dpi=160)\n    plt.close()\n    ranking = pd.DataFrame(summary["model_ranking"])\n    html = ("<!doctype html><html><head><meta charset=\'utf-8\'><title>Laval model report</title>"\n            "<style>body{max-width:1000px;margin:40px auto;font:16px Arial;color:#123}"\n            "td,th{padding:9px;border-bottom:1px solid #ccd}img{max-width:100%}</style></head><body>"\n            "<h1>Laval Random Forest vs analytical</h1><p><b>" + summary["status"] + "</b></p>"\n            "<p>Same held-out covered-driving intervals. These are not whole-lap errors or strategy savings.</p>"\n            + "<p><b>" + summary["recommendation"] + "</b></p>"\n            + ranking.to_html(index=False, float_format=lambda x: f"{x:.2f}")\n            + "<h2>Warnings</h2><ul>" + "".join("<li>" + w + "</li>" for w in summary["warnings"])\n            + "</ul><img src=\'comparison.png\'><h2>Per-session results</h2>"\n            + sessions.to_html(index=False, float_format=lambda x: f"{x:.2f}") + "</body></html>")\n    (out / "report.html").write_text(html, encoding="utf-8")\n\n\ndef run_workflow(archive: str | Path, output: str | Path, cfg: Config,\n                 reference_path: str | Path | None = None, candidates=None) -> dict:\n    cfg.validate()\n    out = Path(output)\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError("Choose an empty/new output directory; existing results are never overwritten")\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")\n    frame, quality, provenance = load_archive(archive, cfg)\n    provenance["implementation_sha256"] = sha256(Path(__file__).read_bytes())\n    quality.to_csv(out / "data_quality.csv", index=False)\n    (out / "source_manifest.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    frame.to_csv(out / "canonical_intervals.csv.gz", index=False, compression="gzip")\n    dev, test, split = split_sessions(frame, cfg)\n    (out / "split_manifest.json").write_text(json.dumps(split, indent=2), encoding="utf-8")\n    selected, cv_scores = tune_forest(dev, cfg, candidates)\n    cv_scores.to_csv(out / "development_cv.csv", index=False)\n    forest = train_forest(dev, selected, cfg)\n    analytical = train_analytical(dev)\n    # Do not inspect test scores until feature/model selection has completed.\n    predictions = {"random_forest": predict_power(forest, test),\n                   "analytical_refit": predict_power(analytical, test)}\n    if reference_path is not None:\n        reference_bytes = Path(reference_path).read_bytes()\n        reference = json.loads(reference_bytes)\n        predictions["frozen_front_campus"] = predict_front_campus(test, reference)\n        (out / "front_campus_reference.json").write_bytes(reference_bytes)\n        provenance["front_campus_reference_sha256"] = sha256(reference_bytes)\n        (out / "source_manifest.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    session_tables, window_tables, prediction_tables = [], [], []\n    ranking = []\n    for name, pred in predictions.items():\n        sessions, windows, rows = evaluation(test, pred, name, cfg)\n        session_tables.append(sessions)\n        window_tables.append(windows)\n        prediction_tables.append(rows)\n        finite = sessions.absolute_error_pct.dropna()\n        if finite.empty:\n            raise ValueError("No positive-energy held-out sessions; percentage errors are undefined")\n        ranking.append({"model": name, "mean_session_error_pct": float(finite.mean()),\n                        "median_session_error_pct": float(finite.median()),\n                        "p90_session_error_pct": float(finite.quantile(0.9)),\n                        "mean_signed_session_error_pct": float(sessions.signed_error_pct.mean()),\n                        "window_energy_mae_wh": float((windows.predicted_energy_j-windows.actual_energy_j).abs().mean()/3600)\n                        if len(windows) else None})\n    sessions = pd.concat(session_tables, ignore_index=True)\n    sessions.to_csv(out / "held_out_sessions.csv", index=False)\n    pd.concat(window_tables, ignore_index=True).to_csv(out / "held_out_windows.csv", index=False)\n    pd.concat(prediction_tables, ignore_index=True).to_csv(out / "held_out_predictions.csv.gz", index=False, compression="gzip")\n    warnings = [\n        "Only September 19-20 Laval data; no untouched future driving day is established.",\n        "Errors cover accepted moving intervals, not whole laps or all recorded session energy.",\n        "No measured throttle channel: this estimator does not establish counterfactual strategy savings.",\n        "The analytical refit uses matched core features; it is not the historical deployment simulator.",\n        "Frozen Front Campus uses zero grade and no additional motor-physics floor; this is observed-driving replay.",\n        "This archive has already been explored in an audit; this is retrospective session-held-out evaluation, not a new prospective field test.",\n        "Repeated tuning after viewing these test errors invalidates the holdout."\n    ]\n    if not cfg.calibration_confirmed:\n        warnings.insert(0, "UNVERIFIED current calibration: all models and errors are exploratory only.")\n    if quality.get("constant_voltage", pd.Series(dtype=bool)).fillna(False).any():\n        warnings.insert(0, "Constant recorded voltage detected: energy labels assume recorded voltage is correct; battery sag is unobserved.")\n    status = "CALIBRATION CONFIRMED BY USER; FIELD VALIDATION STILL REQUIRED" if cfg.calibration_confirmed else "EXPLORATORY - CALIBRATION NOT CONFIRMED"\n    forest.update({"config": asdict(cfg), "source_manifest": provenance, "split_manifest": split,\n                   "status": status, "training_scope": "development_sessions_only"})\n    analytical.update({"config": asdict(cfg), "status": status, "training_scope": "development_sessions_only"})\n    joblib.dump(forest, out / "random_forest_development.joblib")\n    joblib.dump(analytical, out / "analytical_development.joblib")\n    (out / "feature_schema.json").write_text(json.dumps({\n        "version": SCHEMA_VERSION, "target": "trapezoidal interval-average recorded electrical power (W)",\n        "random_forest_features": forest["features"], "analytical_features": CORE_FEATURES,\n        "preprocessing": "causal trailing-3 speed median/history; reset at session/outage; fold-fitted imputation",\n        "inference": "predict_power(bundle, canonical_feature_dataframe); raw telemetry requires prepare_session",\n    }, indent=2), encoding="utf-8")\n    summary = {"status": status, "selected_candidate": selected, "warnings": warnings,\n               "development_sessions": split["development_sessions"], "test_sessions": split["test_sessions"],\n               "accepted_rows": len(frame), "development_rows": len(dev), "test_rows": len(test),\n               "accepted_duration_s": float(frame.dt_s.sum()), "test_duration_s": float(test.dt_s.sum()),\n               "model_ranking": sorted(ranking, key=lambda r: r["mean_session_error_pct"]),\n               "environment": {"python": sys.version, "platform": platform.platform(),\n                               "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,\n                               "joblib": joblib.__version__}}\n    # Explicitly optional and separately named; it has no independent test after this refit.\n    baseline_errors = [r["mean_session_error_pct"] for r in ranking if r["model"] != "random_forest"]\n    rf_error = next(r["mean_session_error_pct"] for r in ranking if r["model"] == "random_forest")\n    improvement = bool(rf_error < min(baseline_errors))\n    summary["forest_beats_all_baselines"] = improvement\n    summary["recommendation"] = "Candidate improves retrospective energy error; obtain a fresh-day test." if improvement else "No improvement over every baseline demonstrated. Keep the best existing baseline."\n    if cfg.refit_all_for_release and improvement:\n        release = train_forest(frame, selected, cfg)\n        release.update({"config": asdict(cfg), "source_manifest": provenance,\n                        "status": "REFIT ON ALL DATA - NEEDS A NEW FUTURE TEST",\n                        "training_scope": "all_sessions_including_previous_test"})\n        joblib.dump(release, out / "random_forest_all_data_UNTESTED.joblib")\n    if cfg.refit_all_for_release and not improvement:\n        summary["warnings"].append("All-data release refit skipped: forest did not beat every analytical baseline.")\n    (out / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")\n    save_report(out, summary, sessions)\n    try:\n        freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)\n    except (OSError, subprocess.CalledProcessError):\n        freeze = "pip freeze unavailable; see summary.json for package versions\\n"\n    (out / "environment.txt").write_text(freeze, encoding="utf-8")\n    # Copy the actual training implementation so results remain usable away from GitHub.\n    (out / "laval_training.py").write_bytes(Path(__file__).read_bytes())\n    with zipfile.ZipFile(str(out) + ".zip", "w", zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(out.iterdir()):\n            if path.is_file():\n                z.write(path, arcname=path.name)\n    print(json.dumps(summary, indent=2, allow_nan=False))\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archive")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--config")\n    parser.add_argument("--reference", help="Optional frozen Front Campus coefficient JSON")\n    args = parser.parse_args()\n    cfg = Config(**json.loads(Path(args.config).read_text())) if args.config else Config()\n    run_workflow(args.archive, args.output, cfg, args.reference)\n\n\nif __name__ == "__main__":\n    main()\n'
BUNDLED_SOURCE_SHA256 = 'ce9dde6c2c644f42ccf34d4daf7d1ecee0a5ae1047d5047753246723f086230e'
BUNDLED_WORKER_SOURCE = '"""Execute all scientific operations outside the interactive notebook kernel."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport sys\nfrom pathlib import Path\n\n\ndef load_training(path):\n    spec = importlib.util.spec_from_file_location("utsm_laval_worker_training", path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("operation", choices=["preview", "train", "check-model"])\n    parser.add_argument("--source", required=True)\n    parser.add_argument("--archive", required=True)\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--reference")\n    args = parser.parse_args()\n    training = load_training(Path(args.source).resolve())\n    cfg = training.Config(**json.loads(Path(args.config).read_text(encoding="utf-8")))\n    cfg.validate()\n    out = Path(args.output).resolve()\n    if args.operation == "train":\n        training.run_workflow(args.archive, out, cfg, args.reference)\n        return\n    if args.operation == "preview":\n        canonical, quality, provenance = training.load_archive(args.archive, cfg)\n        dev, test, split = training.split_sessions(canonical, cfg)\n        out.mkdir(parents=True, exist_ok=True)\n        quality.to_csv(out/"data_quality.csv", index=False)\n        summary = {\n            "source_manifest": provenance, "split_manifest": split,\n            "coverage": [\n                {"split": "development", "sessions": int(dev.session_id.nunique()),\n                 "rows": len(dev), "valid_minutes": float(dev.dt_s.sum()/60)},\n                {"split": "held_out", "sessions": int(test.session_id.nunique()),\n                 "rows": len(test), "valid_minutes": float(test.dt_s.sum()/60)},\n            ],\n            "constant_voltage": bool(quality.get("constant_voltage",\n                                     training.pd.Series(dtype=bool)).fillna(False).any()),\n        }\n        (out/"preview.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n        print("Preview ready:", out)\n        return\n    # Read only saved features/outputs. Loading/predicting never imports NumPy in the parent.\n    import joblib\n    import numpy as np\n    import pandas as pd\n    model = joblib.load(out/"random_forest_development.joblib")\n    canonical = pd.read_csv(out/"canonical_intervals.csv.gz")\n    split = json.loads((out/"split_manifest.json").read_text(encoding="utf-8"))\n    held = canonical[canonical.session_id.isin(split["test_sessions"])]\n    examples = held[model["features"]].head(5)\n    watts = training.predict_power(model, examples)\n    saved = pd.read_csv(out/"held_out_predictions.csv.gz")\n    recorded = saved[saved.model == "random_forest"].predicted_power_w.head(5).to_numpy()\n    np.testing.assert_allclose(watts, recorded)\n    result = {"passed": True, "predictions": json.loads(\n        examples.assign(predicted_power_w=watts).to_json(orient="records"))}\n    (out/"inference_example.json").write_text(json.dumps(result, indent=2), encoding="utf-8")\n    print("Saved-model prediction round trip passed.")\n\n\nif __name__ == "__main__":\n    main()\n'
FRONT_CAMPUS_REFERENCE_JSON = '{\n  "provenance": {\n    "name": "Front Campus 2026-08-06 transferable vehicle model",\n    "manifest": "data/models/front-campus-2026-08-06.json",\n    "run_count": 3,\n    "lap_count": 15,\n    "training_sample_count": 8115,\n    "current_calibration": {\n      "slope": 0.7949,\n      "intercept_mA": 0.0,\n      "note": "Bench slope applied to historical zero-corrected current. Source CSV values remain unchanged."\n    },\n    "sources": [\n      {\n        "run_id": "front-campus-2026-08-06-run-01",\n        "label": "Front Campus 2026-08-06 Run 1",\n        "gpx": "data/runs/front-campus-2026-08-06-run-01/front-campus-2026-08-06-run-01.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-01/front-campus-2026-08-06-run-01-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 6,\n        "loaded_laps": 6,\n        "raw_telemetry_samples": 4272,\n        "merged_training_samples": 3836,\n        "source_negative_current_samples": 3279,\n        "source_over_20a_samples": 506\n      },\n      {\n        "run_id": "front-campus-2026-08-06-run-02",\n        "label": "Front Campus 2026-08-06 Run 2",\n        "gpx": "data/runs/front-campus-2026-08-06-run-02/front-campus-2026-08-06-run-02.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-02/front-campus-2026-08-06-run-02-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 3,\n        "loaded_laps": 3,\n        "raw_telemetry_samples": 1538,\n        "merged_training_samples": 1375,\n        "source_negative_current_samples": 786,\n        "source_over_20a_samples": 129\n      },\n      {\n        "run_id": "front-campus-2026-08-06-run-03",\n        "label": "Front Campus 2026-08-06 Run 3",\n        "gpx": "data/runs/front-campus-2026-08-06-run-03/front-campus-2026-08-06-run-03.gpx",\n        "telemetry": "data/runs/front-campus-2026-08-06-run-03/front-campus-2026-08-06-run-03-telemetry.csv",\n        "split_method": "gate",\n        "expected_laps": 6,\n        "loaded_laps": 6,\n        "raw_telemetry_samples": 3508,\n        "merged_training_samples": 2904,\n        "source_negative_current_samples": 1550,\n        "source_over_20a_samples": 410\n      }\n    ]\n  },\n  "current_coeffs": [\n    90.34341307680654,\n    472.73651713104965,\n    -10.621379963950767,\n    6047.207614491555,\n    -3454.2345120696073,\n    0.0,\n    0.0,\n    0.0,\n    -11161.70737454799,\n    243.4125495281661,\n    10918.294838436992,\n    -68.72024433024941,\n    172.26089666640658\n  ],\n  "power_coeffs": [\n    2.19435097004979,\n    11.482297264596363,\n    -0.25798269794441164,\n    146.88062574838378,\n    -83.899902063659,\n    0.0,\n    0.0,\n    0.0,\n    -271.10671023022945,\n    5.912247605660822,\n    265.1944635209566,\n    -1.6691460145372288,\n    4.184044919130484\n  ],\n  "ridge": 0.001,\n  "on_current_mA": 19490.948,\n  "cruise_current_mA": 64.3869,\n  "median_voltage_v": 24.289,\n  "coast_sample_count": 439,\n  "transferable": true\n}\n'
assert hashlib.sha256(BUNDLED_TRAINING_SOURCE.encode()).hexdigest() == BUNDLED_SOURCE_SHA256
MODULE_PATH = WORK_DIR/"laval_training.py"
MODULE_PATH.write_text(BUNDLED_TRAINING_SOURCE, encoding="utf-8")
WORKER_PATH = WORK_DIR/"laval_notebook_worker.py"
WORKER_PATH.write_text(BUNDLED_WORKER_SOURCE, encoding="utf-8")
REFERENCE_PATH = WORK_DIR/"front_campus_reference.json"
REFERENCE_PATH.write_text(FRONT_CAMPUS_REFERENCE_JSON, encoding="utf-8")
print("Training source:", BUNDLED_SOURCE_SHA256)

def run_worker(operation, output):
    command = [
        sys.executable, "-I", str(WORKER_PATH), operation,
        "--source", str(MODULE_PATH), "--archive", str(archive_path),
        "--config", str(CONFIG_PATH), "--output", str(output),
        "--reference", str(REFERENCE_PATH),
    ]
    subprocess.run(command, check=True)

# Rendering publishes raw HTML; scientific libraries are never used in the kernel.
def show_rows(rows, limit=30):
    import html
    rows = list(rows)
    if not rows:
        print("No rows")
        return
    columns = list(rows[0])
    markup = "<table><thead><tr>" + "".join("<th>"+html.escape(str(c))+"</th>" for c in columns) + "</tr></thead><tbody>"
    for row in rows[:limit]:
        markup += "<tr>" + "".join("<td>"+html.escape(str(row.get(c, "")))+"</td>" for c in columns) + "</tr>"
    markup += "</tbody></table>"
    try:
        display({"text/html": markup}, raw=True)
    except NameError:
        for row in rows[:limit]:
            print(row)


## 3. Calibration and inputs
Leave ARCHIVE_PATH empty to upload, or enter a local/Drive ZIP path.
Defaults are assumptions, not measured calibration. Use bench records to set zero/gain/polarity.
Do not tune quality rules or parameters after viewing September 20 errors.


In [ ]:
ARCHIVE_PATH = os.environ.get("UTSM_ARCHIVE_PATH", "")
USE_GOOGLE_DRIVE = False
SAVE_RESULTS_TO_DRIVE = False
DRIVE_RESULTS_FOLDER = "/content/drive/MyDrive/UTSM-model-results"
CURRENT_MODE = "magnitude"  # magnitude | negative | positive
CURRENT_SCALE = 1.0
CURRENT_ZERO_MA = 0.0
CALIBRATION_CONFIRMED = False
CALIBRATION_OVERRIDES = {}
TEST_DATES = ["2026-09-20"]
N_TREES = int(os.environ.get("UTSM_N_ESTIMATORS", "200"))
REFIT_ALL_FOR_RELEASE = False

config = {
    "current_mode": CURRENT_MODE, "current_scale": CURRENT_SCALE,
    "current_zero_mA": CURRENT_ZERO_MA, "calibration_confirmed": CALIBRATION_CONFIRMED,
    "calibration_overrides": CALIBRATION_OVERRIDES, "test_dates": TEST_DATES,
    "n_estimators": N_TREES, "refit_all_for_release": REFIT_ALL_FOR_RELEASE,
}
CONFIG_PATH = WORK_DIR/"config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
if not CALIBRATION_CONFIRMED:
    print("EXPLORATORY: current calibration is not confirmed.")


## 4. Provide the ZIP
The source archive is read without modification.


In [ ]:
if USE_GOOGLE_DRIVE or SAVE_RESULTS_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is only available in Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
if not ARCHIVE_PATH:
    if not IN_COLAB:
        raise ValueError("Set ARCHIVE_PATH outside Colab.")
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Upload exactly one telemetry ZIP.")
    archive_path = WORK_DIR/Path(names[0]).name
    archive_path.write_bytes(uploaded[names[0]])
else:
    archive_path = Path(ARCHIVE_PATH).expanduser().resolve()
if not archive_path.is_file():
    raise FileNotFoundError(archive_path)
print("Archive:", archive_path.name)


## 5. Preview quality and split
Preparation runs in the worker. Review valid duration, excluded files, current sign and sensor disagreement.
Metrics cover accepted moving intervals, not complete laps or total session energy.


In [ ]:
import csv
preview_dir = WORK_DIR/"preview"
run_worker("preview", preview_dir)
preview = json.loads((preview_dir/"preview.json").read_text(encoding="utf-8"))
with (preview_dir/"data_quality.csv").open(encoding="utf-8", newline="") as stream:
    show_rows(csv.DictReader(stream))
show_rows(preview["coverage"])
print(json.dumps(preview["split_manifest"], indent=2))
print("Archive SHA-256:", preview["source_manifest"]["archive_sha256"])
if preview["constant_voltage"]:
    print("CONSTANT VOLTAGE: labels assume recorded voltage is correct; battery sag is unobserved.")


## 6. Retrain and compare
Session-grouped development tuning chooses a forest before inspecting September 20 errors.
Compare the forest with a matched analytical refit and frozen Front Campus model.
All computation runs in a fresh Python process with the verified on-disk packages.


In [ ]:
from datetime import datetime, timezone
run_name = "laval-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output_dir = Path(os.environ.get("UTSM_OUTPUT_DIR", str(WORK_DIR/"outputs"/run_name))).resolve()
run_worker("train", output_dir)
summary = json.loads((output_dir/"summary.json").read_text(encoding="utf-8"))
print("Decision:", summary["recommendation"])
print("Results:", output_dir)


## 7. Results
Lower error is better. These are retrospective held-out driving-session results.
Independent future-day data and verified calibration are required before claiming race savings.


In [ ]:
show_rows(summary["model_ranking"])
with (output_dir/"held_out_sessions.csv").open(encoding="utf-8", newline="") as stream:
    show_rows(csv.DictReader(stream))
try:
    import base64
    image_data = base64.b64encode((output_dir/"comparison.png").read_bytes()).decode("ascii")
    display({"text/html": '<img style="max-width:100%" src="data:image/png;base64,'+image_data+'">'}, raw=True)
except NameError:
    print("Chart:", output_dir/"comparison.png")
for warning in summary["warnings"]:
    print("-", warning)


## 8. Reload the model and verify predictions
Serialization and inference checks also run in the worker, without importing joblib/NumPy into the kernel.
The prediction inputs exclude current, power and energy labels.


In [ ]:
run_worker("check-model", output_dir)
inference = json.loads((output_dir/"inference_example.json").read_text(encoding="utf-8"))
assert inference["passed"]
show_rows(inference["predictions"])
print("Saved-model prediction round trip passed.")
# Include the checked inference example in the final download.
import zipfile
result_zip = Path(str(output_dir)+".zip")
with zipfile.ZipFile(result_zip, "a", zipfile.ZIP_DEFLATED) as bundle:
    if "inference_example.json" not in bundle.namelist():
        bundle.write(output_dir/"inference_example.json", "inference_example.json")


## 9. Save/download
Save the ZIP before ending the runtime. It contains trained models, preprocessing, metrics, source and reports.
The notebook does not replace the deployed model. A worse forest is not silently promoted.


In [ ]:
import shutil
if SAVE_RESULTS_TO_DRIVE:
    destination = Path(DRIVE_RESULTS_FOLDER)/output_dir.name
    destination.mkdir(parents=True, exist_ok=False)
    shutil.copy2(result_zip, destination/result_zip.name)
    print("Saved to Drive:", destination)
if IN_COLAB:
    from google.colab import files
    files.download(str(result_zip))
else:
    print("Result bundle:", result_zip)


## Next steps
Verify current calibration and fixed voltage first, then add verified lap boundaries and throttle commands.
Keep a future day independent. CPU baselines should justify any later GPU experiments.
The notebook is generated from reviewed source; tests check its embedded files and execution isolation.
